### 문제 2-1 : 콤마 구분 리스트 파서 활용

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_upstage import ChatUpstage

# TODO 1: 파서를 만드세요
parser = CommaSeparatedListOutputParser()

# TODO 2: 파서에게 "형식 지침 문장"을 뽑아달라고 요청하세요
format_instructions = parser.get_format_instructions()

# TODO 3: 프롬프트 템플릿을 만드세요
prompt = PromptTemplate(
    template="한국 {field}의 유명한 장소와 관련 5가지를 알려주세요 \n{format_instructions}",
    input_variables=["field"],
    partial_variables={"format_instructions": format_instructions}
)

# TODO 4: 모델을 만드세요
model = ChatUpstage(model="solar-pro")

# TODO 5: LCEL로 체인을 연결하세요
chain = prompt | model | parser

# TODO 6: 체인을 실행하세요
result = chain.invoke({"field": "음식"})
print(result)
print(type(result))  # 확인용 — 결과가 어떤 자료형인지 보여줍니다

['명동 교자', '남대문 시장 칼국수', '북촌 설렁탕', '광장시장 마복림 떡볶이', '신당동 떡볶이 골목']
<class 'list'>


### 문제 2-2 : 영화 리뷰 감정 분석기

In [20]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.prompts import PromptTemplate
from langchain_upstage import ChatUpstage

# TODO 1: Enum 클래스를 만드세요
class Sentiment(Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"

# TODO 2: 파서를 만드세요
parser = EnumOutputParser(enum=Sentiment)

# TODO 3: 형식 지침을 뽑으세요 (2-1에서 했던 것과 같아요)
format_instructions = parser.get_format_instructions()

model = ChatUpstage(model="solar-pro")

# 기존 parser(EnumOutputParser)를 감싸서, 실패 시 자동 재시도하는 새 파서를 만듭니다
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=model)

# TODO 4: 프롬프트 템플릿
prompt = PromptTemplate(
    template="다음 영화 리뷰의 감정을 분류하세요.\n\n리뷰: {review}\n\n{format_instructions}",
    input_variables=["review"],
    partial_variables={"format_instructions": format_instructions}
)

# TODO 5: LCEL로 체인을 연결하세요
chain = prompt | model | fixing_parser

# 테스트 리뷰 3개
test_reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요."
]

# TODO 6: 반복문으로 각 리뷰를 분석하고 결과를 깔끔하게 출력하세요
for review in test_reviews:
    result = chain.invoke({"review": review})
    print(f"리뷰: {review}")
    print(f"감정: {result.value}")
    print("---")

리뷰: 이 영화 정말 재미없어요. 시간 낭비였습니다.
감정: 부정
---
리뷰: 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
감정: 긍정
---
리뷰: 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
감정: 보통
---
